# Operation participant-account distributions: P1 and P2
Compare `OPERACAO.NUM_CONTA_PARTICIPANTE_P1` and `OPERACAO.NUM_CONTA_PARTICIPANTE_P2` separately against `onprem-export-full`. Counts are operations, not instruments. The IF-account comparison has been removed.

The historical filenames are retained for the deployed application. Upload this notebook and `compare_if_account_distribution.py` together to OCI Data Science, using Python 3.11 and your existing OCI-enabled `spark` session. All outputs are displayed; no report files are written.

**Standalone Data Flow:** the `.py` is the only application code file. Pass `--queries-uri` pointing to the existing `queries_produtos.sql` catalog as runtime data, plus `--source-base-uri` and `--synthetic-run-base-uri`. Omit `--product` for all five products or repeat it for a subset. The default baseline is `product_query_matched`; use `--baseline all` to display all four references and `--top-n 30` to bound each role's preview.

In [ ]:
RUN_ID = '20260911T221522Z-a2ef1130'
EXPORT_BASE = 'oci://oci-st-blc-engordai-qab-n@gr97zovfhcmu/onprem-export-full'
RUN_BASE = f'oci://oci-st-blc-engordai-qab-n@gr97zovfhcmu/pipeline-runs/portuguesa/{RUN_ID}'
PRODUCT_TYPES = {
    'cdb_simplificado': 49,
    'cdb_resgate': 49,
    'cdb_escalonamento': 49,
    'rdb_resgate': 50,
    'rdb_inclusao': 50,
}
PRODUCTS = list(PRODUCT_TYPES)
SYNTHETIC_BASES = {p: f'{RUN_BASE}/products/{p}/synthetic' for p in PRODUCTS}
HELPER_FILE = 'compare_if_account_distribution.py'
QUERIES_URI = 'oci://oci-st-blc-engordai-qab-n@gr97zovfhcmu/scripts/queries_produtos.sql'  # None for legacy-only comparisons.
BASELINE_TO_VIEW = 'product_query_matched'
TOP_N = 30

## Baselines and counting
| Baseline | Source operations included |
| --- | --- |
| `full_export` | Every source operation, including those with NULL or unmatched `NUM_IF`. |
| `same_type` | Operations associated with source IFs of type 49 (CDB) or 50 (RDB), including excluded IFs. |
| `active_same_type` | Operations associated with matching-type IFs whose `DAT_EXCLUSAO IS NULL`. |
| `product_query_matched` | All source operations belonging to IFs returned by the selected product's query in the supplied SQL catalog. |

IF metadata uses only `NUM_IF`, `NUM_TIPO_IF`, and `DAT_EXCLUSAO`; its account column is not compared. The product query reads whichever additional RAW tables/columns its canonical SQL requires, directly from the source export. No product business predicates are independently recreated in the comparison script.

**Scope:** `product_query_matched` reproduces the supplied SQL domain only, not the later Python pruning or live Oracle FK admission. It is not the final admitted population or the exact producing plan's selected input. The catalog URI and SHA-256 values are printed for provenance; a mutable catalog URI does not prove which SQL revision a past generation run used. Keep the catalog and inputs unchanged during the comparison.

The SQL qualifies IF roots. After qualification, all source operations attached to those roots enter this baseline, including operations that did not individually satisfy an existence predicate. No extra status/TOS filter is silently applied to the account aggregation. To run only the three older baselines, set `QUERIES_URI=None` and choose one of them in `BASELINE_TO_VIEW`.

`NUM_ID_OPERACAO` defines the counting grain. Each operation contributes once to P1 and once to P2, but the roles have separate totals: N operations means N observations per role, never a pooled denominator of 2N. All synthetic operations are retained, including those with NULL or unmatched IF references.

`DELTA_PP = SYNTHETIC_PCT - SOURCE_PCT`. Positive values indicate greater synthetic representation. NULL/blank accounts form an explicit NULL bucket. `TOTAL_VARIATION_PCT` is half the sum of absolute percentage-point differences, calculated separately per role and baseline. Empty cohorts have NULL percentages/distances.

These are marginal distributions, not per-operation mutation checks. The IF clone map does not uniquely identify original operations, so the old clone-weighted baselines and per-IF account-change assertions are removed. Identical distributions cannot rule out account swaps between operations.

In [ ]:
from functools import reduce
from pathlib import Path
import runpy
from pyspark import StorageLevel
from pyspark.sql import functions as F

if 'spark' not in globals():
    raise RuntimeError('Use an existing Spark session configured for OCI Object Storage.')
if not PRODUCTS or len(PRODUCTS) != len(set(PRODUCTS)):
    raise ValueError('Select at least one product, without duplicates.')
if any(p not in PRODUCT_TYPES or p not in SYNTHETIC_BASES for p in PRODUCTS):
    raise ValueError('Every selected product needs an IF type and synthetic base URI.')
if BASELINE_TO_VIEW not in {'full_export', 'same_type', 'active_same_type', 'product_query_matched'}:
    raise ValueError(f'Unknown baseline: {BASELINE_TO_VIEW}')
if BASELINE_TO_VIEW == 'product_query_matched' and not QUERIES_URI:
    raise ValueError('product_query_matched requires QUERIES_URI.')
if not isinstance(TOP_N, int) or isinstance(TOP_N, bool) or not 1 <= TOP_N <= 1000:
    raise ValueError('TOP_N must be an integer in 1..1000.')
if not Path(HELPER_FILE).is_file():
    raise FileNotFoundError(f'Upload the comparison helper or adjust HELPER_FILE: {HELPER_FILE}')
helpers = runpy.run_path(HELPER_FILE)
compare_operation_accounts = helpers['compare_operation_accounts']
read_product_query_catalog = helpers['read_product_query_catalog']
product_query_if_ids = helpers['product_query_if_ids']

for frame in globals().get('operation_account_caches', []):
    frame.unpersist()
operation_account_caches = []
operation_account_reports = {}
catalog_text = read_product_query_catalog(spark, QUERIES_URI) if QUERIES_URI else None
operation_columns = [
    'NUM_ID_OPERACAO', 'NUM_IF', 'NUM_CONTA_PARTICIPANTE_P1', 'NUM_CONTA_PARTICIPANTE_P2'
]
try:
    source_operations = spark.read.parquet(EXPORT_BASE.rstrip('/') + '/OPERACAO').select(
        *operation_columns
    ).persist(StorageLevel.MEMORY_AND_DISK)
    operation_account_caches.append(source_operations)
    source_if_metadata = spark.read.parquet(
        EXPORT_BASE.rstrip('/') + '/INSTRUMENTO_FINANCEIRO'
    ).select('NUM_IF', 'NUM_TIPO_IF', 'DAT_EXCLUSAO').persist(StorageLevel.MEMORY_AND_DISK)
    operation_account_caches.append(source_if_metadata)
except Exception:
    for frame in operation_account_caches:
        frame.unpersist()
    operation_account_caches = []
    raise
print(f'Source operations: {EXPORT_BASE}/OPERACAO; run: {RUN_ID}; products: {PRODUCTS}')

In [ ]:
try:
    for product in PRODUCTS:
        base = SYNTHETIC_BASES[product].rstrip('/')
        print(f'Comparing operation accounts for {product}: {base}/OPERACAO')
        synthetic_operations = spark.read.parquet(f'{base}/OPERACAO').select(
            *operation_columns
        ).persist(StorageLevel.MEMORY_AND_DISK)
        operation_account_caches.append(synthetic_operations)
        matched_roots = None
        if catalog_text is not None:
            matched_roots = product_query_if_ids(
                spark, EXPORT_BASE, catalog_text, product, source_if_metadata, PRODUCT_TYPES[product]
            )
            operation_account_caches.append(matched_roots)
        report = compare_operation_accounts(
            source_operations, synthetic_operations, source_if_metadata, PRODUCT_TYPES[product],
            query_matched_ifs=matched_roots
        )
        report['distribution'].persist(StorageLevel.MEMORY_AND_DISK)
        operation_account_caches.append(report['distribution'])
        operation_account_reports[product] = report
        for role in ('P1', 'P2'):
            print(f'{product}: OPERACAO.NUM_CONTA_PARTICIPANTE_{role}')
            report['summary'].where(F.col('ROLE') == role).orderBy('BASELINE').show(
                n=4 if catalog_text is not None else 3, truncate=100
            )
            comparison = report['distribution'].where(
                (F.col('BASELINE') == BASELINE_TO_VIEW) & (F.col('ROLE') == role)
            ).orderBy(F.abs(F.col('DELTA_PP')).desc_nulls_last(), 'NUM_CONTA_PARTICIPANTE')
            comparison.select(
                'ROLE', 'NUM_CONTA_PARTICIPANTE', 'SOURCE_OPERATION_COUNT',
                'SYNTHETIC_OPERATION_COUNT',
                F.round('SOURCE_PCT', 6).alias('SOURCE_PCT'),
                F.round('SYNTHETIC_PCT', 6).alias('SYNTHETIC_PCT'),
                F.round('DELTA_PP', 6).alias('DELTA_PP'), 'PRESENCE'
            ).show(n=TOP_N, truncate=100)
finally:
    for frame in operation_account_caches:
        frame.unpersist()
    operation_account_caches = []
    print('Comparison caches released; the shared Spark session is still running.')

def combine_reports(name):
    frames = [
        report[name].withColumn('PRODUCT', F.lit(product)).withColumn('RUN_ID', F.lit(RUN_ID))
        for product, report in operation_account_reports.items()
    ]
    return reduce(lambda a, b: a.unionByName(b), frames)

operation_account_distribution = combine_reports('distribution')
operation_account_summary = combine_reports('summary')
print('No report files written. The two combined DataFrames remain available for inspection.')